# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "The quick brown fox jumps over the lazy dog near the river bank."
print(sample_sentence)

In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


### Exercise 1 reflection

**[CLS] et [SEP] dans l'encodeur :**
- `[CLS]` (Classification token) est toujours placé en première position. Son embedding de sortie agrège l'information de toute la séquence via l'attention bidirectionnelle — c'est lui qu'on utilise pour les tâches de classification (on branche une tête linéaire dessus).
- `[SEP]` (Separator token) marque la fin d'une phrase (ou la frontière entre deux phrases dans les tâches de paires). Il aide le modèle à comprendre la structure de l'entrée et à ne pas "confondre" les deux segments.

**Attention mask et padding :**
L'attention mask vaut `1` pour les vrais tokens et `0` pour les tokens de padding (`[PAD]`). Lors du calcul des scores d'attention (`QKᵀ`), les positions avec mask=0 reçoivent un score `-inf` avant le softmax, ce qui les réduit à une probabilité d'attention quasi-nulle. Ainsi le modèle ignore complètement les tokens de padding et ne "contamine" pas les représentations des vrais tokens avec de l'information vide.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "The movie was absolutely fantastic, I loved every single minute of it!"
prediction = sentiment_pipeline(sentence)
prediction

### Exercise 2 reflection

**Le label correspond-il à l'attente ?**
Oui — la phrase "The movie was absolutely fantastic, I loved every single minute of it!" est explicitement enthousiaste avec des marqueurs positifs forts ("absolutely fantastic", "loved"). On s'attendait à `POSITIVE` et c'est bien ce que le modèle retourne.

**Que nous dit le score de confiance ?**
DistilBERT fine-tuné sur SST-2 retourne généralement un score > 0.99 sur des phrases aussi clairement positives. Un score proche de 1.0 indique que le modèle est très sûr de sa classification — la distribution softmax est très pointue sur la classe POSITIVE. Un score proche de 0.5 indiquerait une phrase ambiguë ou un sentiment mixte que le modèle peine à trancher.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict


class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.max_length = max_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.labels = ["NEGATIVE", "POSITIVE"]

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        text = text.strip()
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {k: v.to(self.device) for k, v in encoding.items()}

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        predicted_idx = probs.argmax().item()
        return {
            "label": self.labels[predicted_idx],
            "probability": round(probs[predicted_idx].item(), 4),
        }

In [ ]:
analyzer = BERTSentimentAnalyzer()

samples = [
    "This product exceeded all my expectations, absolutely worth every penny!",
    "The service was terrible and the food was cold. Never coming back.",
    "It was okay, nothing special but nothing bad either.",
]

for text in samples:
    result = analyzer.predict(text)
    print(f"Text      : {text}")
    print(f"Sentiment : {result['label']}  (confidence: {result['probability']:.4f})")
    print()

## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch


class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.id2label = self.model.config.id2label

    def recognize(self, text: str):
        encoding = self.tokenizer(
            text,
            return_tensors="pt",
            return_offsets_mapping=True,
            truncation=True,
            max_length=512,
        )
        offset_mapping = encoding.pop("offset_mapping")[0].tolist()
        inputs = {k: v.to(self.device) for k, v in encoding.items()}

        with torch.no_grad():
            logits = self.model(**inputs).logits

        predictions = logits.argmax(dim=-1)[0].tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(encoding["input_ids"][0].tolist())

        entities = []
        current_entity = None

        for token, pred_id, offsets in zip(tokens, predictions, offset_mapping):
            if token in ("[CLS]", "[SEP]", "[PAD]"):
                continue
            label = self.id2label[pred_id]
            start, end = offsets

            if label.startswith("B-"):
                if current_entity:
                    entities.append(current_entity)
                current_entity = {
                    "text": text[start:end],
                    "entity": label[2:],
                    "start": start,
                    "end": end,
                }
            elif label.startswith("I-") and current_entity:
                current_entity["text"] += text[current_entity["end"]:end]
                current_entity["end"] = end
            else:
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None

        if current_entity:
            entities.append(current_entity)

        return entities

In [ ]:
ner = BERTNamedEntityRecognizer()

sample_text = "Elon Musk founded SpaceX in 2002 and Tesla is headquartered in Austin, Texas. He was born in Pretoria, South Africa."

results = ner.recognize(sample_text)
print("Entités détectées :")
for entity in results:
    print(f"  [{entity['entity']}] '{entity['text']}'  (chars {entity['start']}–{entity['end']})")

## Exercise 5 - Comparing BERT and GPT

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-only, bidirectionnel (lit la séquence dans les deux sens) | Decoder-only, autorégressif (lit de gauche à droite uniquement) |
| Primary purpose | Compréhension du langage (représentations contextuelles) | Génération du langage (prédiction du prochain token) |
| Typical use cases | Classification, NER, Q&A extractive, analyse de sentiment, similarité sémantique | Génération de texte, chatbot, complétion de code, résumé génératif |
| Strengths | Contexte bidirectionnel riche, excellent pour la compréhension fine, fine-tuning efficace | Génère du texte fluide et cohérent sur de longues séquences, versatile zero-shot |
| Weaknesses | Ne génère pas de texte nativement (pas autorégressif), contexte limité à 512 tokens | Contexte unidirectionnel limite la compréhension fine, hallucinations fréquentes |

## Exercise 6 - BERT inside Retrieval-Augmented Generation

1. **Comment BERT encode les requêtes et documents :** BERT transforme une requête utilisateur ou un passage de document en un vecteur dense de dimension fixe (ex. 768 pour bert-base). On utilise soit l'embedding `[CLS]` (qui agrège toute la séquence), soit la moyenne des embeddings de tous les tokens (mean pooling). Ce vecteur capture la sémantique du texte indépendamment des mots exacts utilisés — deux phrases avec le même sens mais des mots différents auront des vecteurs proches dans l'espace sémantique.

2. **Stockage et recherche dans une base vectorielle :** Les embeddings de tous les documents du corpus sont précalculés et stockés dans une base vectorielle (Pinecone, Weaviate, ChromaDB, FAISS). À l'arrivée d'une requête, son embedding est calculé en temps réel puis comparé aux vecteurs du corpus via une **recherche de similarité cosinus** ou produit scalaire. Les k passages les plus similaires (k-nearest neighbors approximés) sont retournés en quelques millisecondes même sur des millions de documents.

3. **Transmission des passages récupérés au modèle génératif :** Les k passages les plus pertinents sont concaténés avec la question originale pour former un prompt enrichi : `"Context: [passage1] [passage2] ... Question: [query] Answer:"`. Ce prompt est passé à un modèle génératif (GPT-4, Llama, T5) qui peut alors ancrer sa réponse dans des faits réels plutôt que de se fier uniquement à sa mémoire paramétrique, réduisant les hallucinations.

4. **Exemple concret :** **Support client intelligent** dans une grande banque. Les politiques internes, contrats et FAQ (milliers de documents PDF) sont encodés avec BERT et stockés dans une base vectorielle. Quand un conseiller pose une question sur un produit spécifique ("Quelles sont les conditions de rachat anticipé du produit X ?"), RAG récupère les passages pertinents du contrat et les injecte dans le prompt de GPT-4 qui génère une réponse précise et vérifiable, avec source — là où GPT-4 seul hallucinerait des chiffres.